# Notebook 5 — Evaluation

**Goal:** Honestly assess model quality — where it works, where it doesn't, and what to improve.

## Why evaluation is its own notebook

A single number (MAE = 2.4 min) hides a lot. A model might be excellent for trains that are already running late but terrible for predicting *whether* an on-time train will stay on time. It might work well for the Red Line downtown but poorly for the Blue Line at O'Hare. It might be great at short horizons and terrible at 20 minutes out.

**Good evaluation answers:**
- Is the model better than the baseline *in the scenarios that actually matter*?
- Are its errors systematic (biased) or random (noisy)?
- Is its uncertainty calibrated — when it says "90th percentile is 5 minutes", does 90% of actual data fall below 5 minutes?
- Where does it fail? What should we improve next?

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, f1_score, confusion_matrix
)

sns.set_theme(style='darkgrid')

# Load data + models
df = pd.read_parquet('../data/features.parquet')
df = df.sort_values('snapshot_time')

cutoff = df['snapshot_time'].max() - pd.Timedelta(days=14)
train = df[df['snapshot_time'] <= cutoff].copy()
test  = df[df['snapshot_time'] >  cutoff].copy()

def add_status(df):
    def label(d):
        if d < -1: return 'ahead'
        if d <= 2: return 'on_time'
        return 'behind'
    df = df.copy()
    df['status'] = df['delay_minutes'].apply(label)
    return df

test = add_status(test)
train = add_status(train)

FEATURE_COLS = [
    'is_red', 'is_blue', 'stop_sequence', 'direction',
    'hour', 'dow', 'is_weekend', 'is_peak_am', 'is_peak_pm',
    'minutes_until', 'is_scheduled', 'is_delayed', 'is_faulty',
    'eta_delta_1', 'eta_delta_2',
]

X_test = test[FEATURE_COLS].fillna(0)
y_test = test['delay_minutes']

MODEL_DIR = Path('../ml_models')
if not (MODEL_DIR / 'xgb_regressor.joblib').exists():
    raise FileNotFoundError('Run notebook 04 first to train models.')

reg = joblib.load(MODEL_DIR / 'xgb_regressor.joblib')
p10 = joblib.load(MODEL_DIR / 'xgb_p10.joblib')
p90 = joblib.load(MODEL_DIR / 'xgb_p90.joblib')

test['pred'] = reg.predict(X_test)
test['pred_p10'] = p10.predict(X_test)
test['pred_p90'] = p90.predict(X_test)
test['error'] = test['pred'] - test['delay_minutes']  # positive = predicted too late
test['abs_error'] = test['error'].abs()

LABELS = ['ahead', 'on_time', 'behind']
label_map = joblib.load(MODEL_DIR / 'label_map.joblib')

print(f'Test rows: {len(test):,}')
print(f'Overall MAE: {mean_absolute_error(y_test, test.pred):.3f} min')

## 1. Residual analysis — are errors systematic or random?

**Residuals** = predicted − actual. A well-behaved model has residuals that:
- Are centered near zero (no systematic bias)
- Have constant spread across the prediction range (homoscedasticity)
- Are not structured (no obvious pattern)

If residuals have a pattern, your model is systematically wrong in a way that more/better features could fix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Residual histogram
axes[0].hist(test['error'].clip(-15, 15), bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='orange', linestyle='--')
axes[0].axvline(test['error'].mean(), color='tomato', linestyle='-', label=f'mean={test["error"].mean():.2f}')
axes[0].set_title('Residuals (predicted − actual)')
axes[0].set_xlabel('Error (minutes)')
axes[0].legend()

# Residuals vs predicted value
sample = test.sample(min(3000, len(test)), random_state=42)
axes[1].scatter(sample['pred'].clip(-5, 20), sample['error'].clip(-15, 15),
                alpha=0.15, s=8, color='steelblue')
axes[1].axhline(0, color='orange', linestyle='--')
axes[1].set_title('Residuals vs predicted delay')
axes[1].set_xlabel('Predicted delay (minutes)')
axes[1].set_ylabel('Error (minutes)')

plt.tight_layout()
plt.show()

print(f'Mean error (bias): {test["error"].mean():.3f} min  (should be near 0)')
print(f'Std of errors:     {test["error"].std():.3f} min')

## 2. Performance by prediction horizon

This is one of the most important slices. A model should be most accurate when the train is nearby (0–5 minutes away) and less accurate when it's far away (15–20 minutes).

If accuracy is *flat* across horizons, your model isn't using the real-time signal well. If accuracy *worsens sharply* at short horizons, something's wrong.

In [ ]:
test['horizon_bucket'] = pd.cut(
    test['minutes_until'].clip(0, 30),
    bins=[0, 2, 5, 10, 15, 20, 30],
    labels=['0-2', '2-5', '5-10', '10-15', '15-20', '20-30']
)

horizon_stats = test.groupby('horizon_bucket').apply(
    lambda g: pd.Series({
        'MAE': mean_absolute_error(g['delay_minutes'], g['pred']),
        'n': len(g)
    })
)

fig, ax = plt.subplots(figsize=(9, 4))
horizon_stats['MAE'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('MAE by prediction horizon (minutes until arrival at poll time)')
ax.set_ylabel('MAE (minutes)')
ax.set_xlabel('Horizon bucket')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(horizon_stats)

## 3. Performance by route and time of day

In [ ]:
import pytz
chicago = pytz.timezone('America/Chicago')

test['hour'] = pd.to_datetime(test['snapshot_time']).dt.tz_convert(chicago).dt.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, route, colour in zip(axes, ['Red', 'Blue'], ['#c60c30', '#00a1de']):
    sub = test[test['route'] == route]
    mae_by_hour = sub.groupby('hour').apply(
        lambda g: mean_absolute_error(g['delay_minutes'], g['pred'])
    )
    mae_by_hour.plot(ax=ax, marker='o', color=colour)
    ax.set_title(f'{route} Line — MAE by hour of day')
    ax.set_xlabel('Hour')
    ax.set_ylabel('MAE (minutes)')

plt.tight_layout()
plt.show()

## 4. Calibration check for quantile predictions

**Calibration** means: when the model says "90% chance the delay is ≤ X", it should actually be ≤ X in ~90% of test cases.

A calibration curve plots the *stated* quantile level against the *observed* coverage. A perfectly calibrated model lies on the diagonal y=x.

In [ ]:
# Test multiple quantile levels using our p10 and p90 models
# (In a production setup you'd train models for each level)

quantile_levels = [0.1, 0.9]
pred_quantiles = [test['pred_p10'].values, test['pred_p90'].values]
observed_coverage = [(y_test.values <= pq).mean() for pq in pred_quantiles]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.scatter(quantile_levels, observed_coverage, s=80, zorder=5, color='steelblue')
for q, obs in zip(quantile_levels, observed_coverage):
    ax.annotate(f'p{int(q*100)}: {obs:.2%}', (q, obs), textcoords='offset points', xytext=(10, 5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Stated quantile level')
ax.set_ylabel('Observed coverage')
ax.set_title('Quantile calibration')
ax.legend()
plt.tight_layout()
plt.show()

for q, obs in zip(quantile_levels, observed_coverage):
    bias = obs - q
    verdict = 'overcovering (intervals too wide)' if bias > 0.02 else 'undercovering (intervals too narrow)' if bias < -0.02 else 'well calibrated'
    print(f'p{int(q*100):2d}: stated={q:.2%} observed={obs:.2%} → {verdict}')

## 5. Decile calibration plot

Another way to check calibration: bucket predictions into deciles (lowest 10%, next 10%, ...) and see what the average *actual* delay is in each bucket. If the model is well-calibrated, the actual average should rise monotonically with the predicted average.

In [ ]:
test['pred_decile'] = pd.qcut(test['pred'], q=10, labels=False, duplicates='drop')

decile_stats = test.groupby('pred_decile').agg(
    mean_pred=('pred', 'mean'),
    mean_actual=('delay_minutes', 'mean'),
    count=('delay_minutes', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(decile_stats['mean_pred'], decile_stats['mean_actual'], 'o-', color='steelblue', label='Model')
lo, hi = decile_stats['mean_pred'].min(), decile_stats['mean_pred'].max()
ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect calibration (pred=actual)')
ax.set_title('Decile calibration: mean predicted delay vs mean actual delay')
ax.set_xlabel('Mean predicted delay (minutes)')
ax.set_ylabel('Mean actual delay (minutes)')
ax.legend()
plt.tight_layout()
plt.show()

print('If points hug the diagonal: model is well-calibrated across the range.')
print('If upper deciles fall below the diagonal: model underestimates large delays (common).')

## 6. Where does the model fail most?

Find the patterns in the worst predictions — the top 5% of absolute errors.

In [ ]:
threshold = test['abs_error'].quantile(0.95)
worst = test[test['abs_error'] >= threshold]

print(f'Worst 5% errors (abs_error ≥ {threshold:.2f} min):')
print(f'  Count: {len(worst):,}')
print()
print('Route breakdown:')
print(worst['route'].value_counts())

print('\nHour of day breakdown (when do the worst errors happen?):')
print(worst['hour'].value_counts().sort_index())

print('\nCTA flags in worst errors:')
print(worst[['is_delayed', 'is_scheduled', 'is_faulty']].mean().round(3))

print('\nWorst error direction: positive = predicted too late, negative = predicted too early')
print(worst['error'].describe())

## 7. Full model comparison table

In [ ]:
# Bring back baseline predictions from notebook 3
global_median = train['delay_minutes'].median()

medians = (
    train.groupby(['route', 'station_id', 'hour', 'is_weekend'])['delay_minutes']
    .median().reset_index().rename(columns={'delay_minutes': 'median_delay'})
)
test_m = test.merge(medians, on=['route', 'station_id', 'hour', 'is_weekend'], how='left')
test_m['median_delay'] = test_m['median_delay'].fillna(global_median)

def delay_to_status(d):
    if d < -1: return 'ahead'
    if d <= 2: return 'on_time'
    return 'behind'

rows = []
for name, pred_reg, pred_cls in [
    ('Zero baseline', np.zeros(len(test)), ['on_time']*len(test)),
    ('Historical median', test_m['median_delay'].values, [delay_to_status(d) for d in test_m['median_delay']]),
    ('XGBoost regressor', test['pred'].values, [delay_to_status(d) for d in test['pred']]),
]:
    mae  = mean_absolute_error(y_test, pred_reg)
    rmse = mean_squared_error(y_test, pred_reg, squared=False)
    acc  = accuracy_score(test['status'], pred_cls)
    f1   = f1_score(test['status'], pred_cls, average='macro', labels=LABELS, zero_division=0)
    rows.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'Accuracy': acc, 'Macro F1': f1})

comparison = pd.DataFrame(rows)
comparison = comparison.set_index('Model')
comparison = comparison.style.highlight_min(subset=['MAE', 'RMSE'], color='#2a9d8f')\
                              .highlight_max(subset=['Accuracy', 'Macro F1'], color='#2a9d8f')\
                              .format('{:.3f}')
comparison

## Key takeaways and next steps

### What this evaluation tells you:
1. **Where the model earns its keep** — if XGBoost is only better than the median baseline at long horizons, that's useful to know (and tell users).
2. **Bias direction** — does the model systematically predict too late or too early? That's fixable with a post-hoc bias correction.
3. **Calibration** — if p90 intervals only cover 70% of actuals, users can't trust the uncertainty estimates.
4. **Hard cases** — if worst errors cluster in off-peak hours or at terminal stations, those are your next feature engineering targets.

### What to improve next (Phase 2+):
- **Headway features** — the gap to the preceding/following train is a major predictor we haven't yet used
- **Per-segment travel times** — empirical speed between consecutive stations
- **Service alert text** — disruptions show up in the CTA Customer Alerts API
- **More data** — model quality generally improves with more training examples; run the collector for weeks
- **GTFS schedule matching** — right now actual arrivals are inferred; better GTFS matching → cleaner labels
- **Cross-validation** — instead of a single train/test split, use multiple time-based folds for more robust metrics

## Learning resources

If you want to go deeper on the statistical concepts:
- **Quantile regression**: [Koenker & Bassett (1978)](https://www.jstor.org/stable/1913643) (the original paper) — or just read the [XGBoost docs](https://xgboost.readthedocs.io/en/stable/tutorials/quantile_regression.html)
- **Gradient boosting**: [Friedman (2001)](https://projecteuclid.org/journals/annals-of-statistics/volume-29/issue-5/Greedy-function-approximation-A-gradient-boosting-machine/10.1214/aos/1013203451.full) — or [this blog post](https://explained.ai/gradient-boosting/) for intuition
- **Model calibration**: Platt scaling and isotonic regression are standard techniques to fix miscalibrated probabilities
- **Time-series cross-validation**: sklearn's `TimeSeriesSplit` is the tool for this